<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_16_%D0%9A%D0%B2%D0%B0%D0%BD%D1%82%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BF%D1%80%D0%BE%D0%B3%D0%BD%D0%BE%D0%B7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
import joblib
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from workalendar.europe import Russia
from tqdm import tqdm

In [ ]:
HORIZONS = {
    '4h':  8,
    '8h':  16,
    '24h': 48,
    '7d':  336,
    '14d': 672,
    '1m':  1488,
}

In [ ]:
QUANTILES = [0.1, 0.5, 0.9, 0.95]

In [ ]:
HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12, 'r_ud': 1.2777},
    'house_2': {'n_flats': 191, 'n_floors': 12, 'r_ud': 1.3726},
    'house_3': {'n_flats': 124, 'n_floors': 12, 'r_ud': 1.4664},
    'house_4': {'n_flats': 263, 'n_floors': 12, 'r_ud': 1.3317},
    'house_5': {'n_flats': 127, 'n_floors':  7, 'r_ud': 1.4622},
    'house_6': {'n_flats': 497, 'n_floors': 25, 'r_ud': 1.2506},
    'house_7': {'n_flats': 471, 'n_floors': 17, 'r_ud': 1.2558},
    'house_8': {'n_flats': 171, 'n_floors': 23, 'r_ud': 1.4006},
}

In [ ]:
HOUSES = list(HOUSE_META.keys())

In [ ]:
df = pd.read_csv('df_final+whether.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df['is_holiday'] = df['timestamp'].apply(is_holiday)

p_calc = {h: m['r_ud'] * m['n_flats'] * 1.05 for h, m in HOUSE_META.items()}

p_calc_df = pd.DataFrame([
    {'Дом': h, 'Квартир': HOUSE_META[h]['n_flats'],
     'Руд': HOUSE_META[h]['r_ud'], 'P_расч (кВт)': round(v, 1)}
    for h, v in p_calc.items()
])
print(p_calc_df.to_string(index=False))

    Дом  Квартир    Руд  P_расч (кВт)
house_1      383 1.2777         513.8
house_2      191 1.3726         275.3
house_3      124 1.4664         190.9
house_4      263 1.3317         367.7
house_5      127 1.4622         195.0
house_6      497 1.2506         652.6
house_7      471 1.2558         621.1
house_8      171 1.4006         251.5


In [ ]:
def make_features_all(df, lag_features=[1, 2, 48, 96, 336]):
    frames = []
    for house, meta in HOUSE_META.items():
        data = df[['timestamp', house]].copy()
        data = data.rename(columns={house: 'power'})
        data['hour'] = data['timestamp'].dt.hour
        data['minute'] = data['timestamp'].dt.minute
        data['weekday'] = data['timestamp'].dt.weekday
        data['month'] = data['timestamp'].dt.month
        data['day_of_year'] = data['timestamp'].dt.dayofyear
        data['is_weekend'] = (data['weekday'] >= 5).astype(int)
        data['is_holiday'] = df['is_holiday'].values
        for lag in lag_features:
            data[f'lag_{lag}'] = data['power'].shift(lag)
        data['rolling_mean_48'] = data['power'].shift(1).rolling(48).mean()
        data['rolling_mean_336'] = data['power'].shift(1).rolling(336).mean()
        data['temp_c'] = df['temp_c'].values
        data['humidity'] = df['humidity'].values
        data['cloudiness'] = df['cloudiness'].values
        data['n_flats'] = meta['n_flats']
        data['n_floors'] = meta['n_floors']
        data['house_id'] = house
        frames.append(data)

    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.sort_values(['timestamp', 'house_id']).reset_index(drop=True)
    df_all = df_all.dropna().reset_index(drop=True)
    return df_all

df_long = make_features_all(df)

feature_cols = [c for c in df_long.columns
                if c not in ['timestamp', 'power', 'house_id']]

In [ ]:
def make_features(df, house, lag_features=[1, 2, 48, 96, 336]):
    data = df[['timestamp', house]].copy()

    data['hour'] = data['timestamp'].dt.hour
    data['minute'] = data['timestamp'].dt.minute
    data['weekday'] = data['timestamp'].dt.weekday
    data['month'] = data['timestamp'].dt.month
    data['day_of_year'] = data['timestamp'].dt.dayofyear
    data['is_weekend'] = (data['weekday'] >= 5).astype(int)
    data['is_holiday'] = df['is_holiday'].values

    for lag in lag_features:
        data[f'lag_{lag}'] = data[house].shift(lag)

    data['rolling_mean_48'] = data[house].shift(1).rolling(48).mean()
    data['rolling_mean_336'] = data[house].shift(1).rolling(336).mean()

    data['temp_c'] = df['temp_c'].values
    data['humidity'] = df['humidity'].values
    data['cloudiness'] = df['cloudiness'].values

    data = data.dropna().reset_index(drop=True)
    feature_cols = [c for c in data.columns if c not in ['timestamp', house]]

    return data[feature_cols], data[house], data['timestamp']

X_check, y_check, _ = make_features(df, 'house_1')
print(f'Признаков: {X_check.shape[1]}')
print(f'Строк: {X_check.shape[0]}')
print(f'Список: {X_check.columns.tolist()}')

Признаков: 17
Строк: 35924
Список: ['hour', 'minute', 'weekday', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'rolling_mean_48', 'rolling_mean_336', 'temp_c', 'humidity', 'cloudiness']


In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    err = y_true - y_pred
    return np.mean(np.where(err >= 0, alpha * err, (alpha - 1) * err))

def coverage(y_true, q_low, q_high):
    return np.mean((y_true >= q_low) & (y_true <= q_high)) * 100

def conformal_correction(y_val, pred_val_low, pred_val_high, coverage_target=0.80):
    scores = np.maximum(pred_val_low - y_val, y_val - pred_val_high)
    return np.quantile(scores, coverage_target)

In [ ]:
#обучение общих квантильных моделей
os.makedirs('models_ev', exist_ok=True)

results = {}
results_conformal = {}

for hz_name, hz_steps in tqdm(HORIZONS.items(), desc='Квантильные модели'):
    print(f'\nГоризонт {hz_name}')

    # сдвигаем целевую переменную
    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-hz_steps)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    # split для текущего горизонта
    split_train_h = pd.Timestamp(
        np.quantile(ts_shifted.astype(np.int64), 0.70)
    )
    split_val_h = pd.Timestamp(
        np.quantile(ts_shifted.astype(np.int64), 0.85)
    )

    mask_tr = ts_shifted <= split_train_h
    mask_val = (ts_shifted > split_train_h) & (ts_shifted <= split_val_h)
    mask_te = ts_shifted > split_val_h

    X_train = df_shifted.loc[mask_tr,  feature_cols]
    y_train = df_shifted.loc[mask_tr,  'power_target']
    X_val = df_shifted.loc[mask_val, feature_cols]
    y_val = df_shifted.loc[mask_val, 'power_target']
    X_test = df_shifted.loc[mask_te,  feature_cols]
    y_test = df_shifted.loc[mask_te,  'power_target']
    ts_test = df_shifted.loc[mask_te,  'timestamp']
    house_test = df_shifted.loc[mask_te, 'house_id']

    print(f' train={len(X_train)} val={len(X_val)} test={len(X_test)}')

    # обучаем квантильные модели
    models = {}
    for alpha in QUANTILES:
        model = lgb.LGBMRegressor(
            objective='quantile',
            alpha=alpha,
            metric='quantile',
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )
        models[alpha] = model
        joblib.dump(model, f'models_ev/lgbm_{hz_name}_q{int(alpha*100)}.pkl')
        print(f'  α={alpha:.2f} best_iter={model.best_iteration_}')

    # предсказания на тесте
    preds_test = {alpha: model.predict(X_test) for alpha, model in models.items()}

    results[hz_name] = {
        'models': models,
        'preds': preds_test,
        'y_test': y_test.values,
        'ts_test': ts_test.values,
        'house_test': house_test.values,
    }

    # конформная калибровка
    pred_val_low  = models[0.1].predict(X_val)
    pred_val_high = models[0.9].predict(X_val)
    pred_val_high95 = models[0.95].predict(X_val)

    q_corr_80 = conformal_correction(y_val.values, pred_val_low,
                                      pred_val_high,   0.80)
    q_corr_85 = conformal_correction(y_val.values, pred_val_low,
                                      pred_val_high95, 0.85)

    results_conformal[hz_name] = {
        'y_test': y_test.values,
        'ts_test': ts_test.values,
        'house_test': house_test.values,
        'q10': preds_test[0.1],
        'q50': preds_test[0.5],
        'q90': preds_test[0.9],
        'q95': preds_test[0.95],
        'q90_conf': preds_test[0.9]  + q_corr_80,
        'q10_conf': preds_test[0.1]  - q_corr_80,
        'q95_conf': preds_test[0.95] + q_corr_85,
        'q_corr_80': round(q_corr_80, 3),
        'q_corr_85': round(q_corr_85, 3),
    }
    print(f'  q_corr_80={q_corr_80:.2f} кВт  q_corr_85={q_corr_85:.2f} кВт')

Квантильные модели:   0%|          | 0/6 [00:00<?, ?it/s]


Горизонт 4h
 train=106248 val=22768 test=22760
  α=0.10 best_iter=500
  α=0.50 best_iter=480
  α=0.90 best_iter=486
  α=0.95 best_iter=464


Квантильные модели:  17%|█▋        | 1/6 [00:38<03:13, 38.74s/it]

  q_corr_80=0.69 кВт  q_corr_85=0.67 кВт

Горизонт 8h
 train=106200 val=22760 test=22752
  α=0.10 best_iter=500
  α=0.50 best_iter=500
  α=0.90 best_iter=290
  α=0.95 best_iter=290


Квантильные модели:  33%|███▎      | 2/6 [01:14<02:27, 36.99s/it]

...

  q_corr_80=0.67 кВт  q_corr_85=0.57 кВт

Горизонт 14d
 train=102528 val=21968 test=21968
  α=0.10 best_iter=195
  α=0.50 best_iter=155
  α=0.90 best_iter=136
  α=0.95 best_iter=148


Квантильные модели:  83%|████████▎ | 5/6 [02:44<00:28, 28.70s/it]

  q_corr_80=1.11 кВт  q_corr_85=0.91 кВт

Горизонт 1m
 train=97960 val=20992 test=20984
  α=0.10 best_iter=482
  α=0.50 best_iter=500
  α=0.90 best_iter=349
  α=0.95 best_iter=114


Квантильные модели: 100%|██████████| 6/6 [03:20<00:00, 33.44s/it]

  q_corr_80=2.06 кВт  q_corr_85=2.11 кВт


In [ ]:
#метрики
metrics_rows = []

for hz_name in HORIZONS:
    r = results[hz_name]

    for house in HOUSES:
        mask_h = r['house_test'] == house
        y_h    = r['y_test'][mask_h]

        if len(y_h) == 0:
            continue

        row = {'house': house, 'horizon': hz_name}
        for alpha in QUANTILES:
            row[f'pinball_{alpha}'] = round(
                pinball_loss(y_h, r['preds'][alpha][mask_h], alpha), 4
            )

        mape = np.mean(
            np.abs((y_h - r['preds'][0.5][mask_h]) / (np.abs(y_h) + 1e-8))
        ) * 100
        row['mape_q50'] = round(mape, 3)

        row['coverage_80pct'] = round(
            coverage(y_h, r['preds'][0.1][mask_h], r['preds'][0.9][mask_h]), 1
        )
        row['coverage_85pct'] = round(
            coverage(y_h, r['preds'][0.1][mask_h], r['preds'][0.95][mask_h]), 1
        )
        metrics_rows.append(row)

df_metrics = pd.DataFrame(metrics_rows)

mape_pivot = df_metrics.pivot(index='house', columns='horizon', values='mape_q50')
mape_pivot = mape_pivot[list(HORIZONS.keys())]
mape_pivot.loc['Среднее'] = mape_pivot.mean()
print('MAPE Q0.5 (медианный прогноз), %:')
print(mape_pivot.round(2).to_string())

df_metrics.to_csv('results_quantile_metrics.csv', index=False)

MAPE Q0.5 (медианный прогноз), %:
horizon    4h    8h   24h    7d   14d     1m
house                                       
house_1  5.84  5.18  5.29  5.91  6.56  10.06
house_2  6.80  6.87  6.86  7.14  7.48   8.68
house_3  7.68  7.91  7.44  7.74  8.04   9.28
house_4  6.18  5.80  5.50  5.99  6.95   9.15
house_5  6.61  6.43  6.37  7.29  8.82  11.76
house_6  6.13  6.74  5.38  5.26  6.01   9.21
house_7  6.30  5.88  4.69  5.06  5.43   7.18
house_8  5.48  5.39  5.27  6.17  6.88   7.30
Среднее  6.38  6.28  5.85  6.32  7.02   9.08


In [ ]:
conf_rows = []

for hz_name in HORIZONS:
    r = results_conformal[hz_name]

    for house in HOUSES:
        mask_h = r['house_test'] == house
        y_h = r['y_test'][mask_h]

        if len(y_h) == 0:
            continue

        conf_rows.append({
            'house': house,
            'horizon': hz_name,
            'cov_80_before': round(coverage(y_h, r['q10'][mask_h],
                                             r['q90'][mask_h]), 1),
            'cov_80_after': round(coverage(y_h, r['q10_conf'][mask_h],
                                             r['q90_conf'][mask_h]), 1),
            'cov_85_before': round(coverage(y_h, r['q10'][mask_h],
                                             r['q95'][mask_h]), 1),
            'cov_85_after': round(coverage(y_h, r['q10_conf'][mask_h],
                                             r['q95_conf'][mask_h]), 1),
        })

df_conf = pd.DataFrame(conf_rows)

for col_before, col_after, target in [
    ('cov_80_before', 'cov_80_after', 80),
    ('cov_85_before', 'cov_85_after', 85),
]:
    pivot_before = df_conf.pivot(
        index='house', columns='horizon', values=col_before
    )[list(HORIZONS.keys())]
    pivot_after = df_conf.pivot(
        index='house', columns='horizon', values=col_after
    )[list(HORIZONS.keys())]
    pivot_before.loc['Среднее'] = pivot_before.mean()
    pivot_after.loc['Среднее']  = pivot_after.mean()

    print(f'\nПокрытие {target}% — До поправки:')
    print(pivot_before.round(1).to_string())
    print(f'\nПокрытие {target}% — После поправки:')
    print(pivot_after.round(1).to_string())


Покрытие 80% — До поправки:
horizon    4h    8h   24h    7d   14d    1m
house                                      
house_1  78.6  80.0  76.9  80.6  81.4  69.3
house_2  72.6  73.0  74.1  75.0  77.3  71.6
house_3  73.0  75.3  79.3  78.3  85.8  69.3
house_4  70.8  71.9  76.2  72.0  72.0  59.6
house_5  74.6  74.0  73.2  69.7  66.8  43.1
house_6  73.6  72.2  74.6  78.7  85.1  66.3
house_7  61.2  62.2  70.3  71.9  74.0  65.8
house_8  76.7  76.2  78.1  72.7  73.8  66.5
Среднее  72.6  73.1  75.3  74.9  77.0  63.9

Покрытие 80% — После поправки:
horizon    4h    8h   24h    7d   14d    1m
house                                      
house_1  84.4  86.4  81.2  85.8  88.3  81.9
house_2  85.2  86.8  82.6  86.2  90.3  92.8
house_3  87.2  90.0  87.9  89.4  95.0  95.4
house_4  81.9  83.3  82.7  81.5  82.9  80.0
house_5  86.0  85.9  81.4  80.7  81.6  72.6
house_6  79.6  78.7  78.4  83.1  88.9  77.9
house_7  66.1  69.0  74.5  76.9  80.2  73.4
house_8  85.6  87.2  84.0  82.9  85.7  87.3
Среднее  82.0  

In [ ]:
# доступная мощность
power_rows = []

for house in HOUSES:
    p_r = p_calc[house]
    for hz_name in HORIZONS:
        r = results_conformal[hz_name]
        mask_h = r['house_test'] == house
        y_h = r['y_test'][mask_h]

        if len(y_h) == 0:
            continue

        for scenario, q_key in [('Q0.5', 'q50'),
                                  ('Q0.9_conf', 'q90_conf'),
                                  ('Q0.95_conf', 'q95_conf')]:
            q_pred  = r[q_key][mask_h]
            p_avail = p_r - q_pred

            power_rows.append({
                'house': house,
                'horizon': hz_name,
                'scenario': scenario,
                'P_расч': round(p_r, 1),
                'P_avail_min': round(float(np.min(p_avail)), 1),
            })

df_power = pd.DataFrame(power_rows)
df_power.to_csv('results_ev_capacity.csv', index=False)
print(df_power[df_power['horizon'] == '24h'].to_string(index=False))

  house horizon   scenario  P_расч  P_avail_min
house_1     24h       Q0.5   513.8        407.4
house_1     24h  Q0.9_conf   513.8        403.3
house_1     24h Q0.95_conf   513.8        399.5
house_2     24h       Q0.5   275.3        232.2
house_2     24h  Q0.9_conf   275.3        229.7
house_2     24h Q0.95_conf   275.3        228.2
house_3     24h       Q0.5   190.9        162.6
house_3     24h  Q0.9_conf   190.9        159.7
house_3     24h Q0.95_conf   190.9        159.1
house_4     24h       Q0.5   367.7        298.7
house_4     24h  Q0.9_conf   367.7        293.1
house_4     24h Q0.95_conf   367.7        290.7
house_5     24h       Q0.5   195.0        148.3
house_5     24h  Q0.9_conf   195.0        144.5
house_5     24h Q0.95_conf   195.0        142.9
house_6     24h       Q0.5   652.6        510.5
house_6     24h  Q0.9_conf   652.6        501.5
house_6     24h Q0.95_conf   652.6        501.7
house_7     24h       Q0.5   621.1        460.1
house_7     24h  Q0.9_conf   621.1      

In [ ]:
# Суточный график
profile_rows = []

for house in HOUSES:
    r = results_conformal['24h']
    mask_h = r['house_test'] == house
    ts = pd.to_datetime(r['ts_test'][mask_h])
    p_r = p_calc[house]

    q90_conf = r['q90_conf'][mask_h]
    p_avail  = p_r - q90_conf

    df_profile = pd.DataFrame({'hour': ts.hour, 'p_avail': p_avail})
    hourly = df_profile.groupby('hour')['p_avail'].mean()

    for hour, val in hourly.items():
        profile_rows.append({
            'house': house, 'hour': hour, 'P_avail': round(val, 1)
        })

df_profile_hourly = pd.DataFrame(profile_rows)

fig2 = go.Figure()
for house in HOUSES:
    subset = df_profile_hourly[df_profile_hourly['house'] == house]
    fig2.add_trace(go.Scatter(
        x=[f'{h:02d}:00' for h in subset['hour']],
        y=subset['P_avail'],
        mode='lines+markers', name=house,
        line=dict(width=1.5), marker=dict(size=4)
    ))

fig2.update_layout(
    title='Суточный график доступной мощности по домам, кВт.'
          ' Сценарий Q0.9_conf | Горизонт 24h',
    xaxis_title='Час суток', yaxis_title='Доступная мощность, кВт',
    hovermode='x unified', template='plotly_white',
    height=450, width=700
)
fig2.show()
fig2.write_image('nb15_daily_profile.png', scale=2)

In [ ]:
# загрузка ввода зима/лето
load_rows = []

for house in HOUSES:
    p_r = p_calc[house]
    r = results_conformal['24h']
    mask_h = r['house_test'] == house
    ts = pd.to_datetime(r['ts_test'][mask_h])
    y = r['y_test'][mask_h]

    df_h = pd.DataFrame({'timestamp': ts, 'power': y})
    df_h['date'] = df_h['timestamp'].dt.date
    df_h['month'] = df_h['timestamp'].dt.month

    daily_max = df_h.groupby('date')['power'].max()

    winter_dates = df_h[df_h['month'].isin([12, 1, 2])]['date'].unique()
    summer_dates = df_h[df_h['month'].isin([6, 7, 8])]['date'].unique()

    winter_max = daily_max[daily_max.index.isin(winter_dates)].max()
    summer_min = daily_max[daily_max.index.isin(summer_dates)].min()

    load_rows.append({
        'Дом':                 house,
        'P_расч (кВт)':       round(p_r, 1),
        'Зимний макс. (кВт)': round(winter_max, 1),
        'Загрузка зима (%)':  round(winter_max / p_r * 100, 1),
        'Летний мин. (кВт)':  round(summer_min, 1),
        'Загрузка лето (%)':  round(summer_min / p_r * 100, 1),
        'Резерв зима (кВт)':  round(p_r - winter_max, 1),
        'Резерв лето (кВт)':  round(p_r - summer_min, 1),
    })

df_load = pd.DataFrame(load_rows)
print(df_load.to_string(index=False))
df_load.to_csv('results_load_factor.csv', index=False)

    Дом  P_расч (кВт)  Зимний макс. (кВт)  Загрузка зима (%)  Летний мин. (кВт)  Загрузка лето (%)  Резерв зима (кВт)  Резерв лето (кВт)
house_1         513.8               104.6               20.4                NaN                NaN              409.2                NaN
house_2         275.3                43.8               15.9                NaN                NaN              231.5                NaN
house_3         190.9                30.8               16.2                NaN                NaN              160.1                NaN
house_4         367.7                72.5               19.7                NaN                NaN              295.2                NaN
house_5         195.0                54.9               28.2                NaN                NaN              140.1                NaN
house_6         652.6               145.7               22.3                NaN                NaN              506.9                NaN
house_7         621.1               169.3

In [ ]:
# итоговая сводная таблица
summary_rows = []

for house in HOUSES:
    p_r = p_calc[house]
    r = results_conformal['24h']
    mask_h = r['house_test'] == house

    winter_max = df_load[df_load['Дом'] == house]['Зимний макс. (кВт)'].values[0]
    summer_min = df_load[df_load['Дом'] == house]['Летний мин. (кВт)'].values[0]

    for scenario, q_key in [('Q0.9_conf', 'q90_conf'),
                              ('Q0.95_conf', 'q95_conf')]:
        q_pred  = r[q_key][mask_h]
        p_avail = p_r - q_pred

        summary_rows.append({
            'Дом':                   house,
            'Сценарий':              scenario,
            'P_расч (кВт)':          round(p_r, 1),
            'Зима: макс нагр (кВт)': round(winter_max, 1),
            'Зима: загрузка (%)':    round(winter_max / p_r * 100, 1),
            'Зима: резерв (кВт)':    round(p_r - winter_max, 1),
            'Лето: мин нагр (кВт)':  round(summer_min, 1),
            'Лето: загрузка (%)':    round(summer_min / p_r * 100, 1),
            'Лето: резерв (кВт)':    round(p_r - summer_min, 1),
            'P_avail_min (кВт)':     round(float(np.min(p_avail)), 1),
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv('results_ev_capacity_final.csv', index=False)

for scenario in ['Q0.9_conf', 'Q0.95_conf']:
    subset = df_summary[df_summary['Сценарий'] == scenario].copy()
    subset.loc['Итого'] = [
        '—', scenario,
        round(subset['P_расч (кВт)'].sum(), 1),
        round(subset['Зима: макс нагр (кВт)'].sum(), 1),
        round(subset['Зима: загрузка (%)'].mean(), 1),
        round(subset['Зима: резерв (кВт)'].sum(), 1),
        round(subset['Лето: мин нагр (кВт)'].sum(), 1),
        round(subset['Лето: загрузка (%)'].mean(), 1),
        round(subset['Лето: резерв (кВт)'].sum(), 1),
        round(subset['P_avail_min (кВт)'].sum(), 1),
    ]
    print(f'\nСценарий {scenario}:')
    print(subset.to_string(index=False))


Сценарий Q0.9_conf:
    Дом  Сценарий  P_расч (кВт)  Зима: макс нагр (кВт)  Зима: загрузка (%)  Зима: резерв (кВт)  Лето: мин нагр (кВт)  Лето: загрузка (%)  Лето: резерв (кВт)  P_avail_min (кВт)
house_1 Q0.9_conf         513.8                  104.6                20.4               409.2                   NaN                 NaN                 NaN              403.3
house_2 Q0.9_conf         275.3                   43.8                15.9               231.5                   NaN                 NaN                 NaN              229.7
house_3 Q0.9_conf         190.9                   30.8                16.1               160.1                   NaN                 NaN                 NaN              159.7
house_4 Q0.9_conf         367.7                   72.5                19.7               295.2                   NaN                 NaN                 NaN              293.1
house_5 Q0.9_conf         195.0                   54.9                28.2               140.1     

In [ ]:
# Сводная по горизонтам
rows_hz = []
for house in HOUSES:
    row = {'Дом': house, 'P_расч (кВт)': round(p_calc[house], 1)}
    for hz_name in HORIZONS.keys():
        r      = results_conformal[hz_name]
        mask_h = r['house_test'] == house
        p_avail = p_calc[house] - r['q90_conf'][mask_h]
        row[hz_name] = round(float(np.min(p_avail)), 1) if len(p_avail) > 0 else np.nan
    rows_hz.append(row)

df_hz = pd.DataFrame(rows_hz)
totals = {'Дом': 'Итого', 'P_расч (кВт)': round(df_hz['P_расч (кВт)'].sum(), 1)}
for hz_name in HORIZONS.keys():
    totals[hz_name] = round(df_hz[hz_name].sum(), 1)
df_hz.loc[len(df_hz)] = totals

print(df_hz.to_string(index=False))
df_hz.to_csv('results_ev_all_horizons.csv', index=False)

    Дом  P_расч (кВт)     4h     8h    24h     7d    14d     1m
house_1         513.8  404.5  403.6  403.3  399.6  397.6  405.0
house_2         275.3  230.2  230.9  229.7  229.3  229.8  227.8
house_3         190.9  158.9  159.6  159.7  158.8  159.2  157.2
house_4         367.7  291.2  292.5  293.1  292.1  287.1  290.4
house_5         195.0  142.5  143.7  144.5  142.7  142.3  144.8
house_6         652.6  503.2  505.0  501.5  501.8  505.6  502.5
house_7         621.1  452.1  451.9  451.9  454.7  455.8  462.7
house_8         251.5  188.5  188.4  187.5  187.7  187.8  182.9
  Итого        3067.9 2371.1 2375.6 2371.2 2366.7 2365.2 2373.3


In [ ]:
# сохранение моделей и метаданных
conformal_thresholds = {}
for hz_name in HORIZONS.keys():
    r = results_conformal[hz_name]
    conformal_thresholds[hz_name] = {
        'q_corr_80': r['q_corr_80'],
        'q_corr_85': r['q_corr_85'],
    }
joblib.dump(conformal_thresholds, 'models_ev/conformal_thresholds.pkl')

meta_ev = {
    'houses':       HOUSES,
    'horizons':     HORIZONS,
    'house_meta':   HOUSE_META,
    'p_calc':       p_calc,
    'quantiles':    QUANTILES,
    'feature_cols': feature_cols,
}
joblib.dump(meta_ev, 'models_ev/model_meta.pkl')
print('Сохранено: models_ev/')
print(os.listdir('models_ev'))

Сохранено: models_ev/
['lgbm_8h_q10.pkl', 'lgbm_14d_q10.pkl', 'lgbm_1m_q90.pkl', 'lgbm_7d_q50.pkl', 'lgbm_24h_q95.pkl', 'lgbm_14d_q95.pkl', 'lgbm_4h_q95.pkl', 'lgbm_7d_q90.pkl', 'lgbm_4h_q10.pkl', 'conformal_thresholds.pkl', 'lgbm_14d_q50.pkl', 'model_meta.pkl', 'lgbm_24h_q10.pkl', 'lgbm_4h_q50.pkl', 'lgbm_4h_q90.pkl', 'lgbm_1m_q95.pkl', 'lgbm_24h_q50.pkl', 'lgbm_7d_q10.pkl', 'lgbm_8h_q50.pkl', 'lgbm_8h_q95.pkl', 'lgbm_1m_q10.pkl', 'lgbm_1m_q50.pkl', 'lgbm_14d_q90.pkl', 'lgbm_8h_q90.pkl', 'lgbm_24h_q90.pkl', 'lgbm_7d_q95.pkl']


In [ ]:
# определяем день максимальной и минимальной нагрузки для house_1
r_ref  = results_conformal['24h']
mask_1 = r_ref['house_test'] == 'house_1'
ts_ref = pd.to_datetime(r_ref['ts_test'][mask_1])
y_ref  = r_ref['y_test'][mask_1]

df_ref        = pd.DataFrame({'timestamp': ts_ref, 'power': y_ref})
df_ref['date']  = df_ref['timestamp'].dt.date
df_ref['month'] = df_ref['timestamp'].dt.month

print(f'Месяцы в тесте: {sorted(df_ref["month"].unique())}')
print(f'Период теста: {ts_ref.min().date()} — {ts_ref.max().date()}')

daily_sum  = df_ref.groupby('date')['power'].sum()
winter_day = daily_sum.idxmax()
summer_day = daily_sum.idxmin()

print(f'День макс. нагрузки: {winter_day}')
print(f'День мин. нагрузки:  {summer_day}')

# финальный график house_1
house  = 'house_1'
p_r    = p_calc[house]
r      = results_conformal['24h']
mask_h = r['house_test'] == house

ts     = pd.to_datetime(r['ts_test'][mask_h])
y_test = r['y_test'][mask_h]
q50    = r['q50'][mask_h]
q90    = r['q90_conf'][mask_h]
q10    = r['q10_conf'][mask_h]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'День макс. нагрузки ({winter_day})',
        f'День мин. нагрузки ({summer_day})'
    ],
    horizontal_spacing=0.10
)

for col_idx, target_date in enumerate([winter_day, summer_day], start=1):
    mask_day = np.array([x.date() == target_date for x in ts])
    ts_day   = ts[mask_day].tolist()
    y_day    = y_test[mask_day]
    q50_day  = q50[mask_day]
    q90_day  = q90[mask_day]
    q10_day  = q10[mask_day]
    p_line   = np.full(len(ts_day), p_r)

    fig.add_trace(go.Scatter(
        x=ts_day + ts_day[::-1],
        y=list(p_line) + list(q90_day[::-1]),
        fill='toself',
        fillcolor='rgba(34,139,34,0.10)',
        line=dict(color='rgba(0,0,0,0)'),
        name='Доступная мощность (Q0.9_conf)',
        showlegend=(col_idx == 1), hoverinfo='skip'
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=ts_day + ts_day[::-1],
        y=list(q90_day) + list(q10_day[::-1]),
        fill='toself',
        fillcolor='rgba(200,200,200,0.3)',
        line=dict(color='rgba(0,0,0,0)'),
        name='Интервал Q0.1–Q0.9',
        showlegend=(col_idx == 1), hoverinfo='skip'
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=[ts_day[0], ts_day[-1]], y=[p_r, p_r],
        mode='lines',
        name=f'P_расч = {p_r:.0f} кВт',
        line=dict(color='darkgreen', width=1.5, dash='dash'),
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=ts_day, y=y_day, mode='lines', name='Факт',
        line=dict(color='#1f77b4', width=2, shape='hv'),
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=ts_day, y=q50_day, mode='lines', name='Прогноз Q0.5',
        line=dict(color='#d62728', width=2, shape='hv'),
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)

    fig.update_xaxes(
        tickformat='%H:%M', dtick=3*3600000,
        title_text='Время суток', title_font=dict(size=10),
        tickfont=dict(size=9), row=1, col=col_idx
    )
    fig.update_yaxes(
        range=[0, p_r * 1.05],
        title_text='кВт' if col_idx == 1 else '',
        title_font=dict(size=10), tickfont=dict(size=9),
        row=1, col=col_idx
    )

fig.update_layout(
    title=dict(
        text=f'Дом 1 — квантильный прогноз нагрузки и доступная мощность<br>'
             f'P_расч = {p_r:.0f} кВт | Горизонт 24h | Сценарий Q0.9_conf',
        font=dict(size=12)
    ),
    width=900, height=500,
    font=dict(size=10, family='Arial'),
    margin=dict(l=60, r=30, t=80, b=80),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.18,
        xanchor='center', x=0.5, font=dict(size=10),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='lightgrey', borderwidth=1
    ),
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()
fig.write_image('nb15_house1_winter_summer.png', scale=2)

Месяцы в тесте: [np.int32(10), np.int32(11), np.int32(12)]
Период теста: 2018-10-19 — 2018-12-17
День макс. нагрузки: 2018-11-24
День мин. нагрузки:  2018-12-17


In [ ]:
# P_avail_min по горизонтам
fig_hz = go.Figure()
for house in HOUSES:
    vals = []
    for hz_name in HORIZONS.keys():
        r      = results_conformal[hz_name]
        mask_h = r['house_test'] == house
        p_avail = p_calc[house] - r['q90_conf'][mask_h]
        vals.append(round(float(np.min(p_avail)), 1) if len(p_avail) > 0 else 0)
    fig_hz.add_trace(go.Scatter(
        x=list(HORIZONS.keys()), y=vals,
        mode='lines+markers', name=house,
        line=dict(width=1.5), marker=dict(size=6)
    ))
fig_hz.update_layout(
    title='Гарантированная доступная мощность P_avail_min по горизонтам (Q0.9_conf)',
    xaxis_title='Горизонт прогноза', yaxis_title='P_avail_min, кВт',
    hovermode='x unified', template='plotly_white',
    height=450, width=700
)
fig_hz.show()
fig_hz.write_image('nb15_avail_by_horizon.png', scale=2)

# резерв зима/лето
fig_season = go.Figure()
fig_season.add_trace(go.Bar(
    name='Резерв зима (кВт)', x=df_load['Дом'],
    y=df_load['Резерв зима (кВт)'], marker_color='#1f77b4',
    text=df_load['Резерв зима (кВт)'].round(0), textposition='outside'
))
fig_season.add_trace(go.Bar(
    name='Резерв лето (кВт)', x=df_load['Дом'],
    y=df_load['Резерв лето (кВт)'], marker_color='#ff7f0e',
    text=df_load['Резерв лето (кВт)'].round(0), textposition='outside'
))
fig_season.update_layout(
    title='Резерв мощности под зарядные станции: зима и лето',
    xaxis_title='Дом', yaxis_title='Резерв мощности, кВт',
    barmode='group', template='plotly_white', height=450, width=700
)
fig_season.show()
fig_season.write_image('nb15_season_reserve.png', scale=2)

# загрузка ввода
fig_load = go.Figure()
fig_load.add_trace(go.Bar(
    name='Загрузка зима (%)', x=df_load['Дом'],
    y=df_load['Загрузка зима (%)'], marker_color='#d62728',
    text=[f"{v}%" for v in df_load['Загрузка зима (%)']],
    textposition='outside'
))
fig_load.add_trace(go.Bar(
    name='Загрузка лето (%)', x=df_load['Дом'],
    y=df_load['Загрузка лето (%)'], marker_color='#2ca02c',
    text=[f"{v}%" for v in df_load['Загрузка лето (%)']],
    textposition='outside'
))
fig_load.update_layout(
    title='Загрузка вводного устройства: зима и лето, %',
    xaxis_title='Дом', yaxis_title='Загрузка, %',
    barmode='group', template='plotly_white', height=450, width=700
)
fig_load.show()
fig_load.write_image('nb15_load_factor.png', scale=2)

# график 4 — Q0.9 vs Q0.95
fig_diff = go.Figure()
q90_vals = df_summary[df_summary['Сценарий'] == 'Q0.9_conf']['P_avail_min (кВт)'].values
q95_vals = df_summary[df_summary['Сценарий'] == 'Q0.95_conf']['P_avail_min (кВт)'].values
fig_diff.add_trace(go.Bar(
    name='Q0.9_conf', x=HOUSES, y=q90_vals, marker_color='#1f77b4'
))
fig_diff.add_trace(go.Bar(
    name='Q0.95_conf', x=HOUSES, y=q95_vals, marker_color='#d62728'
))
fig_diff.update_layout(
    title='Сравнение сценариев Q0.9_conf и Q0.95_conf — P_avail_min (горизонт 24h), кВт',
    xaxis_title='Дом', yaxis_title='P_avail_min, кВт',
    barmode='group', template='plotly_white', height=450, width=700
)
fig_diff.show()
fig_diff.write_image('nb15_q90_vs_q95.png', scale=2)